<a href="https://colab.research.google.com/github/saurav2006-cyber/001/blob/main/ML_Lab_09_Supervised_learning_1_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Load dataset
data = fetch_california_housing()

# Convert to DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Price'] = data.target

# Display info
print("Shape:", df.shape)
print("Features:", df.columns)
print(df.head())
print(df.dtypes)
print(df.describe())

Shape: (20640, 9)
Features: Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'Price'],
      dtype='object')
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  Price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  
MedInc        float64
HouseAge      float64
AveRooms      float64
AveBedrms     float64
Population    float64
AveOccup      float64
Latitude      float64
Longitude     float64
Price         float64
dtype: object
             MedInc      Ho

TASK 2: Data Preprocessing

In [5]:
# Missing values
print(df.isnull().sum())

# Duplicates
print(df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

# Split
X = df.drop("Price", axis=1)
y = df["Price"]

MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
Price         0
dtype: int64
0


TASK 3: Train-Test Split

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

4: Feature Scaling

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

5: Train Models

In [8]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "DecisionTree": DecisionTreeRegressor(),
    "KNN": KNeighborsRegressor()
}

results = {}

6: Model Evaluation

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results[name] = [mae, mse, rmse, r2]

7: Comparison Table

In [10]:
df_results = pd.DataFrame(results, index=["MAE", "MSE", "RMSE", "R2"]).T
print(df_results)

                   MAE       MSE      RMSE        R2
Linear        0.533200  0.555892  0.745581  0.575788
Ridge         0.533193  0.555855  0.745557  0.575816
Lasso         0.906069  1.310696  1.144856 -0.000219
DecisionTree  0.457255  0.504082  0.709987  0.615325
KNN           0.446154  0.432422  0.657588  0.670010


In [11]:
ranked = df_results.sort_values(by="R2", ascending=False)
print(ranked)

                   MAE       MSE      RMSE        R2
KNN           0.446154  0.432422  0.657588  0.670010
DecisionTree  0.457255  0.504082  0.709987  0.615325
Ridge         0.533193  0.555855  0.745557  0.575816
Linear        0.533200  0.555892  0.745581  0.575788
Lasso         0.906069  1.310696  1.144856 -0.000219


8: Hyperparameter Tuning

In [12]:
depths = [3, 5, 10, 15]
best_score = -1

for d in depths:
    model = DecisionTreeRegressor(max_depth=d)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = r2_score(y_test, y_pred)

    print(f"Depth {d}: R2 = {score}")

    if score > best_score:
        best_score = score
        best_depth = d

print("Best Depth:", best_depth)

Depth 3: R2 = 0.5097629887358219
Depth 5: R2 = 0.5997321244428705
Depth 10: R2 = 0.6777353171999347
Depth 15: R2 = 0.6406153363008344
Best Depth: 10


9: Cross Validation

In [13]:
from sklearn.model_selection import cross_val_score

model = DecisionTreeRegressor(max_depth=best_depth)

scores = cross_val_score(model, X, y, cv=5)

print("Scores:", scores)
print("Mean:", scores.mean())
print("Std:", scores.std())

Scores: [0.35845766 0.55843975 0.61321041 0.27331292 0.52891242]
Mean: 0.46646663452376913
Std: 0.12873737880634423


10: Feature Engineering

In [14]:
df['rooms_per_household'] = df['AveRooms'] / df['HouseAge']
df['population_per_household'] = df['Population'] / df['AveOccup']

11: Outlier Removal

In [15]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)

IQR = Q3 - Q1

df_clean = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

12: Final Pipeline

In [16]:
import joblib
joblib.dump(model, "best_model.pkl")

['best_model.pkl']

In [17]:
df_results.to_csv("results.csv")